In [23]:
import argparse
import csv
import json
import os
import matplotlib
matplotlib.use("Agg")  # no display needed; we only save files
import matplotlib.pyplot as plt
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.metrics.sequence_labeling import get_entities
from datasets import load_dataset
import sklearn_crfsuite
import re
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
import pandas as pd
from collections import Counter

In [4]:
LABELS_SHORT = ["PER", "ORG", "LOC", "MISC"]
METHODS = ["CRF", "Transformer"]

In [5]:
def load_data():
    ds = load_dataset(
    "eriktks/conll2003",
    revision="refs/convert/parquet"
)
    names = ds["train"].features["ner_tags"].feature.names

    def convert(split):
        return [
            {"tokens": ex["tokens"], "tags": [names[t] for t in ex["ner_tags"]]}
            for ex in split
        ]

    return ds, names, convert(ds["train"]), convert(ds["validation"]), convert(ds["test"])

In [6]:
def word2features(sent, i):
    w = sent[i]
    f = {
        "bias": 1.0,
        "lower": w.lower(),
        "suffix3": w[-3:],
        "prefix3": w[:3],
        "is_upper": w.isupper(),
        "is_title": w.istitle(),
        "is_digit": w.isdigit(),
        "has_hyphen": "-" in w,
    }
    if i > 0:
        p = sent[i - 1]
        f.update({"-1:lower": p.lower(), "-1:is_title": p.istitle(), "-1:is_upper": p.isupper()})
    else:
        f["BOS"] = True
    if i < len(sent) - 1:
        n = sent[i + 1]
        f.update({"+1:lower": n.lower(), "+1:is_title": n.istitle(), "+1:is_upper": n.isupper()})
    else:
        f["EOS"] = True
    return f


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

In [7]:
def run_crf(train, test):
    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
    crf.fit([sent2features(s["tokens"]) for s in train], [s["tags"] for s in train])
    return crf.predict([sent2features(s["tokens"]) for s in test])


In [8]:
def run_transformer(ds, names, test, out_dir, model_name="bert-base-cased", epochs=3):
    from transformers import (
        AutoModelForTokenClassification,
        AutoTokenizer,
        DataCollatorForTokenClassification,
        Trainer,
        TrainingArguments,
    )

    id2label = {i: n for i, n in enumerate(names)}
    label2id = {n: i for i, n in id2label.items()}
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=len(names), id2label=id2label, label2id=label2id
    )

    def tokenize_and_align(batch):
        enc = tok(batch["tokens"], is_split_into_words=True, truncation=True)
        all_labels = []
        for i, tags in enumerate(batch["ner_tags"]):
            prev, aligned = None, []
            for wid in enc.word_ids(batch_index=i):
                if wid is None:
                    aligned.append(-100)  # special tokens
                elif wid != prev:
                    aligned.append(tags[wid])  # first subword gets the label
                else:
                    aligned.append(-100)  # other subwords ignored
                prev = wid
            all_labels.append(aligned)
        enc["labels"] = all_labels
        return enc

    cols = ds["train"].column_names
    tokenized = ds.map(tokenize_and_align, batched=True, remove_columns=cols)

    def decode(logits, labels):
        preds = np.argmax(logits, axis=-1)
        out_pred, out_gold = [], []
        for p_row, l_row in zip(preds, labels):
            out_pred.append([id2label[p] for p, l in zip(p_row, l_row) if l != -100])
            out_gold.append([id2label[l] for l in l_row if l != -100])
        return out_pred, out_gold

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        pred, gold = decode(logits, labels)
        return {"f1": f1_score(gold, pred)}

    args = TrainingArguments(
        output_dir=os.path.join(out_dir, "hf_tmp"),
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=5e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        seed=42,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=DataCollatorForTokenClassification(tok),
        compute_metrics=compute_metrics,
    )
    trainer.train()

    # Package the final model (last epoch; no checkpoint selection) and its tokenizer
    final_dir = os.path.join(out_dir, "final_model")
    trainer.save_model(final_dir)
    tok.save_pretrained(final_dir)

    # The test set is only touched here, once, after training
    out = trainer.predict(tokenized["test"])
    pred, _ = decode(out.predictions, out.label_ids)
    # Pad with "O" in case truncation dropped trailing words (rare in CoNLL)
    pred = [p + ["O"] * (len(s["tags"]) - len(p)) for p, s in zip(pred, test)]

    # Record the configuration next to the model
    from importlib.metadata import version
    from seqeval.scheme import IOB2

    gold_tags = [s["tags"] for s in test]
    config = {
        "base_model": model_name,
        "dataset": "eriktks/conll2003",
        "tag_scheme": "IOB2",
        "labels": list(names),
        "epochs": epochs,
        "learning_rate": args.learning_rate,
        "weight_decay": args.weight_decay,
        "train_batch_size": args.per_device_train_batch_size,
        "seed": args.seed,
        "checkpoint_selection": "none (final epoch)",
        "train_sentences": len(tokenized["train"]),
        "test_metrics_strict": {
            "precision": float(precision_score(gold_tags, pred, mode="strict", scheme=IOB2)),
            "recall": float(recall_score(gold_tags, pred, mode="strict", scheme=IOB2)),
            "f1": float(f1_score(gold_tags, pred, mode="strict", scheme=IOB2)),
        },
        "versions": {p: version(p) for p in ["transformers", "torch", "seqeval", "datasets"]},
    }
    with open(os.path.join(final_dir, "run_config.json"), "w") as f:
        json.dump(config, f, indent=2)

    return pred, trainer.state.log_history

In [9]:
def save_predictions(out_dir, method, test, pred):
    path = os.path.join(out_dir, f"predictions_{method.lower()}.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        for s, p in zip(test, pred):
            f.write(json.dumps({"tokens": s["tokens"], "gold": s["tags"], "pred": list(p)}) + "\n")


def load_predictions(out_dir, method):
    path = os.path.join(out_dir, f"predictions_{method.lower()}.jsonl")
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [10]:
def evaluate(rows):
    gold = [r["gold"] for r in rows]
    pred = [r["pred"] for r in rows]
    return {
        "n_sentences": len(rows),
        "precision": precision_score(gold, pred),
        "recall": recall_score(gold, pred),
        "f1": f1_score(gold, pred),
        "text_report": classification_report(gold, pred, digits=3),
        "per_label": classification_report(gold, pred, digits=3, output_dict=True),
    }


def save_errors(rows, path):
    """CSV of sentences where predicted entities differ from gold entities."""

    def fmt(r, ents):
        return "; ".join(
            f"{' '.join(r['tokens'][s:e + 1])} [{t}]" for t, s, e in sorted(ents, key=lambda x: x[1])
        )

    n_err = 0
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["sentence_idx", "text", "gold_entities", "predicted_entities", "missed", "spurious"])
        for i, r in enumerate(rows):
            g, p = set(get_entities(r["gold"])), set(get_entities(r["pred"]))
            if g == p:
                continue
            n_err += 1
            w.writerow([i, " ".join(r["tokens"]), fmt(r, g), fmt(r, p), fmt(r, g - p), fmt(r, p - g)])
    return n_err



In [11]:
def plot_overall(summary_rows, n_sub, path):
    names = [r["method"] for r in summary_rows]
    metrics = [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]
    x = np.arange(len(names))
    w = 0.26
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for i, (key, label) in enumerate(metrics):
        bars = ax.bar(x + (i - 1) * w, [r[key] for r in summary_rows], w, label=label)
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score (strict, entity-level)")
    ax.set_title(f"CoNLL-2003: full test set ({n_sub} sentences)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=4, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


def plot_per_label(results_sub, path):
    methods = list(results_sub)
    x = np.arange(len(LABELS_SHORT))
    w = 0.8 / max(len(methods), 1)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for i, m in enumerate(methods):
        vals = [results_sub[m]["per_label"].get(lab, {}).get("f1-score", 0.0) for lab in LABELS_SHORT]
        bars = ax.bar(x + (i - (len(methods) - 1) / 2) * w, vals, w, label=m)
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS_SHORT)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("F1")
    ax.set_title("Per-label F1 (strict, entity-level)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=4, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


def plot_confusion(rows, title, path):
    """Token-level confusion over entity types (B-/I- merged). Rows = gold, cols = predicted."""
    cats = ["O"] + LABELS_SHORT
    idx = {c: i for i, c in enumerate(cats)}

    def typ(tag):
        return "O" if tag == "O" else tag.split("-", 1)[1]

    cm = np.zeros((len(cats), len(cats)))
    for r in rows:
        for g, p in zip(r["gold"], r["pred"]):
            cm[idx[typ(g)], idx.get(typ(p), 0)] += 1
    norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(norm, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(cats)))
    ax.set_xticklabels(cats)
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels(cats)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Gold")
    ax.set_title(title)
    for i in range(len(cats)):
        for j in range(len(cats)):
            ax.text(j, i, f"{norm[i, j]:.2f}\n({int(cm[i, j])})", ha="center", va="center",
                    fontsize=7, color="white" if norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, label="Row-normalised")
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


def plot_training(history, path):
    losses = [(h["step"], h["loss"]) for h in history if "loss" in h]
    evals = [(h["epoch"], h["eval_f1"]) for h in history if "eval_f1" in h]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    if losses:
        axes[0].plot(*zip(*losses))
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Training loss")
    axes[0].set_title("Training loss")
    if evals:
        axes[1].plot(*zip(*evals), marker="o")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dev F1")
    axes[1].set_title("Validation F1")
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)

In [12]:
def write_outputs(out_dir, all_rows):
    """all_rows: {method: list of {"tokens","gold","pred"}} for every method with saved predictions."""
    n_test = len(next(iter(all_rows.values())))
    summary, per_label_rows, results = [], [], {}

    for method, rows in all_rows.items():
        res = evaluate(rows)
        with open(os.path.join(out_dir, f"report_{method.lower()}_full_test.txt"), "w") as f:
            f.write(f"{method} | full_test | {res['n_sentences']} sentences\n\n{res['text_report']}")
        summary.append({
            "method": method, "scope": "full_test", "n_sentences": res["n_sentences"],
            "precision": round(res["precision"], 4), "recall": round(res["recall"], 4),
            "f1": round(res["f1"], 4),
        })
        print(f"{method:12s} full_test      P={res['precision']:.3f} R={res['recall']:.3f} F1={res['f1']:.3f}")
        results[method] = res
        for lab in LABELS_SHORT:
            d = res["per_label"].get(lab, {})
            per_label_rows.append({
                "method": method, "label": lab,
                "precision": round(d.get("precision", 0.0), 4),
                "recall": round(d.get("recall", 0.0), 4),
                "f1": round(d.get("f1-score", 0.0), 4),
                "support": int(d.get("support", 0)),
            })
        n_err = save_errors(rows, os.path.join(out_dir, f"errors_{method.lower()}.csv"))
        print(f"  -> {n_err} sentences with errors saved to errors_{method.lower()}.csv")
        plot_confusion(rows, f"{method}: token-level confusion",
                       os.path.join(out_dir, f"confusion_{method.lower()}.png"))

    # Tables
    with open(os.path.join(out_dir, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    for name, rows_ in [("metrics_summary.csv", summary), ("per_label_metrics.csv", per_label_rows)]:
        with open(os.path.join(out_dir, name), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows_[0].keys()))
            w.writeheader()
            w.writerows(rows_)

    with open(os.path.join(out_dir, "summary.md"), "w") as f:
        f.write("| Method | Scope | Sentences | Precision | Recall | F1 |\n|---|---|---|---|---|---|\n")
        for r in summary:
            f.write(f"| {r['method']} | {r['scope']} | {r['n_sentences']} | "
                    f"{r['precision']:.3f} | {r['recall']:.3f} | {r['f1']:.3f} |\n")

    # Charts
    plot_overall(summary, n_test, os.path.join(out_dir, "comparison_overall.png"))
    plot_per_label(results, os.path.join(out_dir, "comparison_per_label.png"))


In [18]:
def main():
    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--steps",
        nargs="+",
        default=["crf", "transformer"],
        choices=["crf", "transformer"]
    )

    ap.add_argument("--out-dir", default="results")
    ap.add_argument("--hf-model", default="bert-base-cased")
    ap.add_argument("--epochs", type=int, default=3)

    # Ignore Jupyter/Colab arguments such as -f kernel.json
    args, unknown = ap.parse_known_args()

    if unknown:
        print("Ignoring unknown arguments:", unknown)

    os.makedirs(args.out_dir, exist_ok=True)

    ds, names, train, dev, test = load_data()

    print(f"train={len(train)}  dev={len(dev)}  test={len(test)}")

    if "crf" in args.steps:
        print("Training CRF...")
        save_predictions(
            args.out_dir,
            "CRF",
            test,
            run_crf(train, test)
        )

    if "transformer" in args.steps:
        print("Fine-tuning transformer...")

        pred, history = run_transformer(
            ds,
            names,
            test,
            args.out_dir,
            args.hf_model,
            args.epochs
        )

        save_predictions(args.out_dir, "Transformer", test, pred)

        with open(
            os.path.join(args.out_dir, "transformer_training_log.json"),
            "w"
        ) as f:
            json.dump(history, f, indent=2)

        plot_training(
            history,
            os.path.join(args.out_dir, "transformer_training.png")
        )

    # Combine predictions saved by this run or previous runs
    all_rows = {
        m: r
        for m in METHODS
        if (r := load_predictions(args.out_dir, m))
    }

    if not all_rows:
        print("No predictions found; nothing to summarise.")
        return

    print("\n=== Results (strict, entity-level) ===")

    write_outputs(args.out_dir, all_rows)

    print(f"\nAll results saved to: {os.path.abspath(args.out_dir)}/")


if __name__ == "__main__":
    main()

Ignoring unknown arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-78e0329a-b0c0-4b41-ade7-f6c32616d4d6.json']


conll2003/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

conll2003/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

...

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1
1,0.045619,0.037232,0.935592
2,0.020711,0.036395,0.945787
3,0.006473,0.036173,0.948110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== Results (strict, entity-level) ===
CRF          full_test      P=0.822 R=0.776 F1=0.798
  -> 1082 sentences with errors saved to errors_crf.csv
Transformer  full_test      P=0.902 R=0.917 F1=0.910
  -> 505 sentences with errors saved to errors_transformer.csv

All results saved to: /content/results/


In [19]:
OUT_DIR = "results"


def load_rows(method):
    rows = load_predictions(OUT_DIR, method)
    if rows is None:
        raise FileNotFoundError(f"No predictions for {method} in {OUT_DIR}/. Run the pipeline first.")
    return rows


def fmt_ents(r, ents):
    return "; ".join(f"{' '.join(r['tokens'][s:e + 1])} [{t}]" for t, s, e in sorted(ents, key=lambda x: x[1]))


def overlaps(a, b):
    return a[1] <= b[2] and b[1] <= a[2]  # entities are (type, start, end), end inclusive


def categorize_errors(gold_tags, pred_tags):
    """Return [(error_kind, type, start, end)] for one sentence.
    For missed/wrong_type/boundary errors the type is the GOLD type; for spurious it is the predicted type."""
    gold, pred = set(get_entities(gold_tags)), set(get_entities(pred_tags))
    errors = []
    for g in gold - pred:
        hits = [p for p in pred if overlaps(g, p)]
        if any(p[1:] == g[1:] for p in hits):
            kind = "wrong_type"        # right span, wrong label
        elif any(p[0] == g[0] for p in hits):
            kind = "boundary"          # right label, wrong span
        elif hits:
            kind = "boundary+type"     # overlapping, wrong span AND wrong label
        else:
            kind = "missed"            # nothing predicted there
        errors.append((kind, *g))
    for p in pred - gold:
        if not any(overlaps(p, g) for g in gold):
            errors.append(("spurious", *p))  # predicted entity with no gold counterpart
    return errors


def error_table(method):
    c = Counter()
    for r in load_rows(method):
        for kind, label, *_ in categorize_errors(r["gold"], r["pred"]):
            c[(kind, label)] += 1
    kinds = ["missed", "wrong_type", "boundary", "boundary+type", "spurious"]
    t = pd.DataFrame({lab: [c[(k, lab)] for k in kinds] for lab in LABELS_SHORT}, index=kinds)
    t["total"] = t.sum(axis=1)
    return t


def show_examples(method, kind, k=6, label=None):
    shown = 0
    for i, r in enumerate(load_rows(method)):
        errs = [e for e in categorize_errors(r["gold"], r["pred"]) if e[0] == kind and (label is None or e[1] == label)]
        if not errs:
            continue
        print(f"[{method} #{i}] {' '.join(r['tokens'])}")
        print("   gold:", fmt_ents(r, get_entities(r["gold"])))
        print("   pred:", fmt_ents(r, get_entities(r["pred"])))
        shown += 1
        if shown >= k:
            break

for m in METHODS:
    print(f"\n=== {m}: error counts (rows = error kind, cols = entity class) ===")
    print(error_table(m))

for kind in ["missed", "wrong_type", "boundary", "spurious"]:
    print(f"\n--- Transformer: {kind} ---")
    show_examples("Transformer", kind, k=5)

err_bert = pd.read_csv(os.path.join(OUT_DIR, "errors_transformer.csv"))
print("\nerrors_transformer.csv:", len(err_bert), "sentences with errors")
print(err_bert[["text", "missed", "spurious"]].head(10).to_string())

crf, bert = load_rows("CRF"), load_rows("Transformer")

def exact(r):
    return set(get_entities(r["gold"])) == set(get_entities(r["pred"]))


fixed = [i for i, (a, b) in enumerate(zip(crf, bert)) if not exact(a) and exact(b)]
broken = [i for i, (a, b) in enumerate(zip(crf, bert)) if exact(a) and not exact(b)]
print(f"\nSentences CRF got wrong but BERT got right: {len(fixed)}")
print(f"Sentences CRF got right but BERT got wrong: {len(broken)}")

gain, loss = Counter(), Counter()
for a, b in zip(crf, bert):
    gold = set(get_entities(a["gold"]))
    pa, pb = set(get_entities(a["pred"])), set(get_entities(b["pred"]))
    for ent in gold:
        if ent in pb and ent not in pa:
            gain[ent[0]] += 1
        if ent in pa and ent not in pb:
            loss[ent[0]] += 1
print("Gold entities BERT gets that CRF misses, by class:", dict(gain))
print("Gold entities CRF gets that BERT misses, by class:", dict(loss))


def show_comparison(idx_list, k=6):
    for i in idx_list[:k]:
        print(f"#{i}: {' '.join(crf[i]['tokens'])}")
        print("   gold:", fmt_ents(crf[i], get_entities(crf[i]["gold"])))
        print("   CRF :", fmt_ents(crf[i], get_entities(crf[i]["pred"])))
        print("   BERT:", fmt_ents(bert[i], get_entities(bert[i]["pred"])))


print("\n--- BERT fixed these ---")
show_comparison(fixed)
print("\n--- BERT broke these ---")
show_comparison(broken)



=== CRF: error counts (rows = error kind, cols = entity class) ===
               PER  ORG  LOC  MISC  total
missed          92  147  143    72    454
wrong_type     108  259  114    59    540
boundary        24   97    8    26    155
boundary+type   15   40   47    14    116
spurious        37   76   20    26    159

=== Transformer: error counts (rows = error kind, cols = entity class) ===
               PER  ORG  LOC  MISC  total
missed          15   20   12    22     69
wrong_type      42   76   60    48    226
boundary        15   37    9    39    100
boundary+type    3   28   27    13     71
spurious        18   52   16    76    162

--- Transformer: missed ---
[Transformer #22] RUGBY UNION - CUTTITTA BACK FOR ITALY AFTER A YEAR .
   gold: RUGBY UNION [ORG]; CUTTITTA [PER]; ITALY [LOC]
   pred: RUGBY UNION [ORG]; CUTTITTA [LOC]
[Transformer #145] Blinker missed his club 's last two games after FIFA slapped a worldwide ban on him for appearing to sign contracts for both Wednesday

In [20]:
# ============================================================================
# Step 2: Per-class precision / recall / F1 (strict, entity-level)
# ============================================================================
from seqeval.scheme import IOB2


def per_class_table(strict=True):
    kw = dict(mode="strict", scheme=IOB2) if strict else {}
    records = []
    for m in METHODS:
        rows = load_rows(m)
        gold, pred = [r["gold"] for r in rows], [r["pred"] for r in rows]
        rep = classification_report(gold, pred, output_dict=True, **kw)
        for lab in LABELS_SHORT + ["micro avg", "macro avg"]:
            d = rep[lab]
            records.append({"method": m, "label": lab, "precision": d["precision"],
                            "recall": d["recall"], "f1": d["f1-score"], "support": int(d["support"])})
    return pd.DataFrame(records)


pc = per_class_table(strict=True)
table = pc.pivot(index="label", columns="method", values=["precision", "recall", "f1"])
table = table.reindex(LABELS_SHORT + ["micro avg", "macro avg"]).round(3)
print(table)
print("\nSupport (gold entities per class):")
print(pc[pc.method == METHODS[0]].set_index("label")["support"].reindex(LABELS_SHORT + ["micro avg"]))
pc.round(4).to_csv(os.path.join(OUT_DIR, "per_class_strict.csv"), index=False)


          precision             recall                 f1            
method          CRF Transformer    CRF Transformer    CRF Transformer
label                                                                
PER           0.831       0.961  0.852       0.953  0.841       0.957
ORG           0.773       0.886  0.673       0.902  0.719       0.894
LOC           0.866       0.937  0.813       0.935  0.839       0.936
MISC          0.802       0.781  0.756       0.825  0.779       0.802
micro avg     0.822       0.909  0.776       0.917  0.798       0.913
macro avg     0.818       0.892  0.774       0.904  0.795       0.897

Support (gold entities per class):
label
PER          1617
ORG          1661
LOC          1668
MISC          702
micro avg    5648
Name: support, dtype: int64


In [21]:
rows_by_method = {m: load_rows(m) for m in METHODS}
ref_name = METHODS[0]
ref = rows_by_method[ref_name]

# 3a) same test sentences, same gold labels, aligned lengths for every model
allowed = {"O"} | {f"{p}-{l}" for p in "BI" for l in LABELS_SHORT}
for m, rows in rows_by_method.items():
    assert len(rows) == len(ref), f"{m}: {len(rows)} sentences vs {len(ref)} for {ref_name}"
    assert all(a["tokens"] == b["tokens"] for a, b in zip(rows, ref)), f"{m}: tokens differ from {ref_name}"
    assert all(a["gold"] == b["gold"] for a, b in zip(rows, ref)), f"{m}: gold labels differ from {ref_name}"
    assert all(len(r["tokens"]) == len(r["gold"]) == len(r["pred"]) for r in rows), f"{m}: length mismatch"
    used = {t for r in rows for t in r["gold"] + r["pred"]}
    assert used <= allowed, f"{m}: unexpected tags {used - allowed}"
print(f"OK: {len(METHODS)} models, same {len(ref)} test sentences, same gold labels, valid tag set.")

# 3b) the saved sentences really are the official test split
ds, names, train, dev, test = load_data()
assert [s["tokens"] for s in test] == [r["tokens"] for r in ref], "predictions are not on the test split"
assert [s["tags"] for s in test] == [r["gold"] for r in ref]
print("OK: predictions line up with the CoNLL-2003 test split.")

# 3c) how many test sentences also appear verbatim in train/dev? (informational)
seen = {tuple(s["tokens"]) for s in train} | {tuple(s["tokens"]) for s in dev}
dups = sum(tuple(s["tokens"]) in seen for s in test)
print(f"Test sentences that also occur verbatim in train/dev: {dups} of {len(test)}")


# 3d) strict scoring: seqeval's DEFAULT mode is lenient about invalid I- starts,
#     mode="strict" with scheme=IOB2 only counts well-formed IOB2 entities.
def invalid_i_starts(tags):
    n, prev = 0, "O"
    for t in tags:
        if t.startswith("I-") and (prev == "O" or prev[2:] != t[2:]):
            n += 1
        prev = t
    return n


print(f"\n{'method':12s} {'default F1':>10s} {'strict F1':>10s} {'invalid I- starts':>18s}")
for m, rows in rows_by_method.items():
    gold, pred = [r["gold"] for r in rows], [r["pred"] for r in rows]
    f_default = f1_score(gold, pred)
    f_strict = f1_score(gold, pred, mode="strict", scheme=IOB2)
    bad = sum(invalid_i_starts(p) for p in pred)
    print(f"{m:12s} {f_default:10.4f} {f_strict:10.4f} {bad:18d}")
print("\nIf default and strict differ, evaluate() is not truly strict. To fix it, pass")
print("mode='strict', scheme=IOB2 to precision_score/recall_score/f1_score/classification_report in evaluate().")

OK: 2 models, same 3453 test sentences, same gold labels, valid tag set.
OK: predictions line up with the CoNLL-2003 test split.
Test sentences that also occur verbatim in train/dev: 133 of 3453

method       default F1  strict F1  invalid I- starts
CRF              0.7983     0.7983                  0
Transformer      0.9098     0.9127                 45

If default and strict differ, evaluate() is not truly strict. To fix it, pass
mode='strict', scheme=IOB2 to precision_score/recall_score/f1_score/classification_report in evaluate().


In [22]:
MODEL_DIR = os.path.join(OUT_DIR, "final_model")
_WORD_RE = re.compile(r"\w+|[^\w\s]")  # words and single punctuation marks, like CoNLL tokens


def load_ner_model(model_dir=MODEL_DIR, device=None):
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    model.to(device or ("cuda" if torch.cuda.is_available() else "cpu")).eval()
    return tok, model


def extract_entities(text, tok, model, max_length=512):
    """Return [{"text", "label", "start", "end"}], with character offsets into `text` (end exclusive)."""
    matches = list(_WORD_RE.finditer(text))
    if not matches:
        return []
    words = [m.group() for m in matches]
    enc = tok(words, is_split_into_words=True, truncation=True, max_length=max_length, return_tensors="pt")
    device = next(model.parameters()).device
    with torch.no_grad():
        logits = model(**{k: v.to(device) for k, v in enc.items()}).logits[0]
    pred_ids = logits.argmax(-1).tolist()

    # Same rule as training: the first sub-token of each word carries the word's label
    word_tags, seen = ["O"] * len(words), set()
    for tok_i, wid in enumerate(enc.word_ids(0)):
        if wid is not None and wid not in seen:
            seen.add(wid)
            word_tags[wid] = model.config.id2label[pred_ids[tok_i]]

    entities, cur = [], None
    for tag, m in zip(word_tags, matches):
        if tag == "O":
            if cur:
                entities.append(cur)
            cur = None
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B" or cur is None or cur["label"] != label:
            if cur:
                entities.append(cur)
            cur = {"label": label, "start": m.start(), "end": m.end()}
        else:
            cur["end"] = m.end()
    if cur:
        entities.append(cur)
    for e in entities:
        e["text"] = text[e["start"]:e["end"]]
    return entities


if os.path.isdir(MODEL_DIR):
    with open(os.path.join(MODEL_DIR, "run_config.json")) as f:
        print(json.dumps(json.load(f), indent=2))
    tok, model = load_ner_model()
    example = "Barack Obama visited Cairo and met officials from the United Nations on Monday."
    for e in extract_entities(example, tok, model):
        print(e)
else:
    print(f"{MODEL_DIR} not found: re-run the transformer step first.")


{
  "base_model": "bert-base-cased",
  "dataset": "eriktks/conll2003",
  "tag_scheme": "IOB2",
  "labels": [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-MISC",
    "I-MISC"
  ],
  "epochs": 3,
  "learning_rate": 5e-05,
  "weight_decay": 0.01,
  "train_batch_size": 16,
  "seed": 42,
  "checkpoint_selection": "none (final epoch)",
  "train_sentences": 14041,
  "test_metrics_strict": {
    "precision": 0.908580452711002,
    "recall": 0.9167847025495751,
    "f1": 0.9126641403014013
  },
  "versions": {
    "transformers": "5.18.0",
    "torch": "2.11.0+cu130",
    "seqeval": "1.2.2",
    "datasets": "4.8.5"
  }
}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'label': 'PER', 'start': 0, 'end': 12, 'text': 'Barack Obama'}
{'label': 'LOC', 'start': 21, 'end': 26, 'text': 'Cairo'}
{'label': 'ORG', 'start': 54, 'end': 68, 'text': 'United Nations'}


-----